In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS de_mini_project.gold")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE de_mini_project.gold.fact_inventory AS
WITH latest_sales AS (
    SELECT 
        product_id, 
        store_id, 
        -- sum sales in last 30 days(this is done to identify whether product is sold within last 30 days or not)
        MAX(transaction_date) as last_sale_date,
        SUM(CASE WHEN transaction_date >= date_sub(current_date(), 30) THEN sold_qty ELSE 0 END) as sales_last_30_days
    FROM de_mini_project.gold.fact_sales
    GROUP BY product_id, store_id
)
SELECT 
    i.sku_id as product_id,
    i.store_id,
    i.stocks_qty,
    i.last_audit_date,
    -- mark whether item is out of stock or not
    CASE WHEN i.stocks_qty = 0 THEN 1 ELSE 0 END AS is_out_of_stock,
    -- mark whether item is sold within last 30 days or not
    CASE WHEN COALESCE(s.sales_last_30_days, 0) = 0 THEN 1 ELSE 0 END AS is_slow_moving,
    s.last_sale_date
FROM de_mini_project.silver.inventory i
LEFT JOIN latest_sales s 
    ON i.sku_id = s.product_id 
    AND i.store_id = s.store_id
""")

In [0]:
display(spark.sql("SELECT store_id, SUM(is_out_of_stock) as oos_count FROM de_mini_project.gold.fact_inventory GROUP BY store_id"))